# データ保存先

In [ ]:
import os
os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# THEMIS-Aの電場・磁場データのdownload

In [ ]:
import pyspedas as psp
import pytplot as pt

pt.del_data('*')

time_range = ['20220901/20:00:00', '20220902/00:00:00']

path_base_save_plot = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/2230-2330'

psp.themis.fgm(trange=time_range, probe='a', level='l2', no_update=True, get_support_data=True)                 # fgh: 128 Hz, fgl: 16 Hz, fgs: 2.74 sec
psp.themis.efi(trange=time_range, probe='a', level='l2', datatype='efp', no_update=True, get_support_data=True) # efp: 512 Hz
psp.themis.efi(trange=time_range, probe='a', level='l2', datatype='efi', no_update=True, get_support_data=True) # eff: 8 Hz
print("--- Loaded tplot variables ---")
print(pt.tplot_names())

In [ ]:
Espin_data_dsl  = pt.data_quants['tha_efs_dot0_dsl']
E8_data_dsl     = pt.data_quants['tha_eff_dot0_dsl']
E512_data_dsl   = pt.data_quants['tha_efp_dsl']
Bspin_data_dsl  = pt.data_quants['tha_fgs_dsl']
B16_data_dsl    = pt.data_quants['tha_fgl_dsl']
B128_data_dsl   = pt.data_quants['tha_fgh_dsl']

time_range_T    = [time_range[0].replace('/', 'T'), time_range[1].replace('/', 'T')]
Espin_data_dsl  = Espin_data_dsl.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all')
E8_data_dsl     = E8_data_dsl.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all')
E512_data_dsl   = E512_data_dsl.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all')
Bspin_data_dsl  = Bspin_data_dsl.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all')
B16_data_dsl    = B16_data_dsl.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all')
B128_data_dsl   = B128_data_dsl.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1])).dropna(dim='time', how='all')

In [ ]:
import xarray as xr
import numpy as np

def split_by_gap(ds, time_dim='time', gap_thr=np.timedelta64(30, 's'),
                 prefix='ds_8_gsm_seg'):
    t = ds[time_dim].values
    if t.size == 0:
        return []

    # 先頭はギャップなし。以降で gap_thr を超えたら新セグ開始。
    gaps = np.r_[False, (t[1:] - t[:-1]) > gap_thr]      # shape (Nt,)
    seg_id = np.cumsum(gaps)                              # 0,0,0,1,1,2,...

    ds_tagged = ds.assign_coords(_seg=(time_dim, seg_id))
    segs = [g.drop_vars('_seg') for _, g in ds_tagged.groupby('_seg')]

    return segs

In [ ]:
ds_Espin_data_dsl = xr.Dataset(
    data_vars={
        "Espin_dsl_x": (("time",), Espin_data_dsl[:, 0].values),
        "Espin_dsl_y": (("time",), Espin_data_dsl[:, 1].values),
        "Espin_dsl_z": (("time",), Espin_data_dsl[:, 2].values),
    },
    coords={
        "time": Espin_data_dsl["time"].values,
    },
)
ds_E8_data_dsl = xr.Dataset(
    data_vars={
        "E8_dsl_x": (("time",), E8_data_dsl[:, 0].values),
        "E8_dsl_y": (("time",), E8_data_dsl[:, 1].values),
        "E8_dsl_z": (("time",), E8_data_dsl[:, 2].values),
    },
    coords={
        "time": E8_data_dsl["time"].values,
    },
)
ds_E512_data_dsl = xr.Dataset(
    data_vars={
        "E512_dsl_x": (("time",), E512_data_dsl[:, 0].values),
        "E512_dsl_y": (("time",), E512_data_dsl[:, 1].values),
        "E512_dsl_z": (("time",), E512_data_dsl[:, 2].values),
    },
    coords={
        "time": E512_data_dsl["time"].values,
    },
)
ds_Bspin_data_dsl = xr.Dataset(
    data_vars={
        "Bspin_dsl_x": (("time",), Bspin_data_dsl[:, 0].values),
        "Bspin_dsl_y": (("time",), Bspin_data_dsl[:, 1].values),
        "Bspin_dsl_z": (("time",), Bspin_data_dsl[:, 2].values),
    },
    coords={
        "time": Bspin_data_dsl["time"].values,
    },
)
ds_B16_data_dsl = xr.Dataset(
    data_vars={
        "B16_dsl_x": (("time",), B16_data_dsl[:, 0].values),
        "B16_dsl_y": (("time",), B16_data_dsl[:, 1].values),
        "B16_dsl_z": (("time",), B16_data_dsl[:, 2].values),
    },
    coords={
        "time": B16_data_dsl["time"].values,
    },
)
ds_B128_data_dsl = xr.Dataset(
    data_vars={
        "B128_dsl_x": (("time",), B128_data_dsl[:, 0].values),
        "B128_dsl_y": (("time",), B128_data_dsl[:, 1].values),
        "B128_dsl_z": (("time",), B128_data_dsl[:, 2].values),
    },
    coords={
        "time": B128_data_dsl["time"].values,
    },
)

In [ ]:
def uniq_and_sort_time(ds):
    idx = ds.get_index('time')
    mask = ~idx.duplicated()          # 最初の出現だけ True
    return ds.isel(time=mask).sortby('time')

In [ ]:
ds_Espin_data_dsl   = uniq_and_sort_time(ds_Espin_data_dsl)
ds_E8_data_dsl      = uniq_and_sort_time(ds_E8_data_dsl)
ds_E512_data_dsl    = uniq_and_sort_time(ds_E512_data_dsl)
ds_Bspin_data_dsl   = uniq_and_sort_time(ds_Bspin_data_dsl)
ds_B16_data_dsl     = uniq_and_sort_time(ds_B16_data_dsl)
ds_B128_data_dsl    = uniq_and_sort_time(ds_B128_data_dsl)

In [ ]:
ds_Espin_data_dsl_segs = split_by_gap(ds_Espin_data_dsl, gap_thr=np.timedelta64(11, 's'))
for ds_ in ds_Espin_data_dsl_segs:
    print(ds_.time)

In [ ]:
ds_E8_data_dsl_segs = split_by_gap(ds_E8_data_dsl, gap_thr=np.timedelta64(500, 'ms'))
for ds_ in ds_E8_data_dsl_segs:
    print(ds_.time)

In [ ]:
ds_E512_data_dsl_segs = split_by_gap(ds_E512_data_dsl, gap_thr=np.timedelta64(8, 'ms'))
for ds_ in ds_E512_data_dsl_segs:
    print(ds_.time)

In [ ]:
ds_Bspin_data_dsl_segs = split_by_gap(ds_Bspin_data_dsl, gap_thr=np.timedelta64(11, 's'))
for ds_ in ds_Bspin_data_dsl_segs:
    print(ds_.time)

In [ ]:
ds_B16_data_dsl_segs = split_by_gap(ds_B16_data_dsl, gap_thr=np.timedelta64(250, 'ms'))
for ds_ in ds_B16_data_dsl_segs:
    print(ds_.time)

In [ ]:
ds_B128_data_dsl_segs = split_by_gap(ds_B128_data_dsl, gap_thr=np.timedelta64(250, 'ms'))
for ds_ in ds_B128_data_dsl_segs:
    print(ds_.time)

# 電場データと磁場データの時間を合わせる

In [ ]:
def make_ds_EB_func(ds_E, E_vars, ds_B, B_vars, time_base, output_vars):
    ds_E_interp = ds_E.interp(time=time_base, method='linear')
    ds_B_interp = ds_B.interp(time=time_base, method='linear')

    da_Ex   = ds_E_interp[E_vars[0]]
    da_Ey   = ds_E_interp[E_vars[1]]
    da_Ez   = ds_E_interp[E_vars[2]]
    da_Bx   = ds_B_interp[B_vars[0]]
    da_By   = ds_B_interp[B_vars[1]]
    da_Bz   = ds_B_interp[B_vars[2]]

    ds_EB   = xr.Dataset({
        output_vars[0]: da_Ex,
        output_vars[1]: da_Ey,
        output_vars[2]: da_Ez,
        output_vars[3]: da_Bx,
        output_vars[4]: da_By,
        output_vars[5]: da_Bz,
    })

    ds_EB   = ds_EB.dropna(dim='time', how='any')

    return ds_EB

In [ ]:
Espin_vars  = ['Espin_dsl_x', 'Espin_dsl_y', 'Espin_dsl_z']
E8_vars     = ['E8_dsl_x', 'E8_dsl_y', 'E8_dsl_z']
E512_vars   = ['E512_dsl_x', 'E512_dsl_y', 'E512_dsl_z']
Bspin_vars  = ['Bspin_dsl_x', 'Bspin_dsl_y', 'Bspin_dsl_z']
B16_vars    = ['B16_dsl_x', 'B16_dsl_y', 'B16_dsl_z']
B128_vars   = ['B128_dsl_x', 'B128_dsl_y', 'B128_dsl_z']
EBspin_vars = ['Espin_dsl_x', 'Espin_dsl_y', 'Espin_dsl_z', 'Bspin_dsl_x', 'Bspin_dsl_y', 'Bspin_dsl_z']
EB8_vars    = ['E8_dsl_x', 'E8_dsl_y', 'E8_dsl_z', 'B8_dsl_x', 'B8_dsl_y', 'B8_dsl_z']
EB128_vars  = ['E128_dsl_x', 'E128_dsl_y', 'E128_dsl_z', 'B128_dsl_x', 'B128_dsl_y', 'B128_dsl_z']

In [ ]:
ds_EBspin_dsl_segs  = []

ds_EBspin_dsl_segs.append(make_ds_EB_func(ds_Espin_data_dsl_segs[0], Espin_vars, ds_Bspin_data_dsl_segs[0], Bspin_vars, ds_Espin_data_dsl_segs[0].time, EBspin_vars).dropna(dim='time', how='all'))

print(ds_EBspin_dsl_segs)

In [ ]:
ds_EB8_dsl_segs = []

ds_EB8_dsl_segs.append(make_ds_EB_func(ds_E8_data_dsl_segs[0], E8_vars, ds_B16_data_dsl_segs[0], B16_vars, ds_E8_data_dsl_segs[0].time, EB8_vars).dropna(dim='time', how='all'))
ds_EB8_dsl_segs.append(make_ds_EB_func(ds_E8_data_dsl_segs[1], E8_vars, ds_B16_data_dsl_segs[2], B16_vars, ds_E8_data_dsl_segs[1].time, EB8_vars).dropna(dim='time', how='all'))
ds_EB8_dsl_segs.append(make_ds_EB_func(ds_E8_data_dsl_segs[2], E8_vars, ds_B16_data_dsl_segs[2], B16_vars, ds_E8_data_dsl_segs[2].time, EB8_vars).dropna(dim='time', how='all'))

print(ds_EB8_dsl_segs)

In [ ]:
ds_EB128_dsl_segs = []

ds_EB128_dsl_segs.append(make_ds_EB_func(ds_E512_data_dsl_segs[0], E512_vars, ds_B128_data_dsl_segs[0], B128_vars, ds_B128_data_dsl_segs[0].time, EB128_vars).dropna(dim='time', how='all'))
ds_EB128_dsl_segs.append(make_ds_EB_func(ds_E512_data_dsl_segs[1], E512_vars, ds_B128_data_dsl_segs[1], B128_vars, ds_B128_data_dsl_segs[1].time, EB128_vars).dropna(dim='time', how='all'))
ds_EB128_dsl_segs.append(make_ds_EB_func(ds_E512_data_dsl_segs[2], E512_vars, ds_B128_data_dsl_segs[2], B128_vars, ds_B128_data_dsl_segs[2].time, EB128_vars).dropna(dim='time', how='all'))

print(ds_EB128_dsl_segs)

In [ ]:
path_base_save_plot = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01_second/2230-2330'
os.makedirs(path_base_save_plot, exist_ok=True)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams['font.size'] = 15

# ---- 5分刻みの時間窓 ----
t_start = np.datetime64('2022-09-01T20:00:00')
t_end   = np.datetime64('2022-09-02T00:00:00')
step_min    = 1
step    = np.timedelta64(step_min, 'm')

t_list = []
t0 = t_start
while t0 < t_end:
    t1 = t0 + step
    t_list.append((t0, t1))
    t0 = t1

def plot_EB_window_all_segs(ds_list, t0, t1, save_dir):
    """全 seg を重ねて [t0, t1] の波形を描いて保存"""

    fig = plt.figure(figsize=(10, 10))
    gs = fig.add_gridspec(6, 1)
    ax_0 = fig.add_subplot(gs[0, 0])
    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
    ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
    ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    ax_1.tick_params(axis='x', which='both', labelbottom=False)
    ax_2.tick_params(axis='x', which='both', labelbottom=False)
    ax_3.tick_params(axis='x', which='both', labelbottom=False)
    ax_4.tick_params(axis='x', which='both', labelbottom=False)

    # ---- 各 seg を同じ軸に重ね描き ----
    for ds_seg in ds_list:
        ds_win = ds_seg.sel(time=slice(t0, t1))
        if ds_win.time.size == 0:
            continue

        ax_0.plot(ds_win.time, ds_win['Espin_dsl_x'], lw=1, c='k')
        ax_1.plot(ds_win.time, ds_win['Espin_dsl_y'], lw=1, c='k')
        ax_2.plot(ds_win.time, ds_win['Espin_dsl_z'], lw=1, c='k')
        ax_3.plot(ds_win.time, ds_win['Bspin_dsl_x'], lw=1, c='k')
        ax_4.plot(ds_win.time, ds_win['Bspin_dsl_y'], lw=1, c='k')
        ax_5.plot(ds_win.time, ds_win['Bspin_dsl_z'], lw=1, c='k')

    ax_0.set_ylabel(r'$E_{x}$ (DSL)' + '\n' + '[mV/m]')
    ax_1.set_ylabel(r'$E_{y}$ (DSL)' + '\n' + '[mV/m]')
    ax_2.set_ylabel(r'$E_{z}$ (DSL)' + '\n' + '[mV/m]')
    ax_3.set_ylabel(r'$B_{x}$ (DSL)' + '\n' + '[nT]')
    ax_4.set_ylabel(r'$B_{y}$ (DSL)' + '\n' + '[nT]')
    ax_5.set_ylabel(r'$B_{z}$ (DSL)' + '\n' + '[nT]')

    for ax in (ax_0, ax_1, ax_2, ax_3, ax_4, ax_5):
        ax.minorticks_on()
        ax.grid(which='both', alpha=0.5)

    ax_5.set_xlim(t0, t1)
    fig.tight_layout()

    if os.path.isdir(save_dir):
        save_dir_   = f'{save_dir}/waveform/{step_min}min_dsl_spin'
        os.makedirs(save_dir_, exist_ok=True)
        t0_str = np.datetime_as_string(t0, unit='m').replace('-', '').replace('T', '_').replace(':', '')
        t1_str = np.datetime_as_string(t1, unit='m').split('T')[1].replace(':', '')
        fname = f'EB_fields_dsl_{t0_str}_{t1_str}.png'
        fpath = os.path.join(save_dir_, fname)
        fig.savefig(fpath)
        plt.close(fig)
    else:
        plt.show()
        plt.close()


from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp

def worker(args):
    """並列実行するためのラッパー"""
    ds_list, t0, t1, save_dir = args
    plot_EB_window_all_segs(ds_list, t0, t1, save_dir)
    return str(t0)  # ログ用

# 並列実行用のタスクをまとめる
tasks = [(ds_EBspin_dsl_segs, t0, t1, path_base_save_plot) for (t0, t1) in t_list]

# CPUコア数
n_cores = max(1, mp.cpu_count())

with ProcessPoolExecutor(max_workers=n_cores) as exe:
    for out in exe.map(worker, tasks):
        print("done:", out)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams['font.size'] = 15

# ---- 5分刻みの時間窓 ----
t_start = np.datetime64('2022-09-01T20:00:00')
t_end   = np.datetime64('2022-09-02T00:00:00')
step_min    = 1
step    = np.timedelta64(step_min, 'm')

t_list = []
t0 = t_start
while t0 < t_end:
    t1 = t0 + step
    t_list.append((t0, t1))
    t0 = t1

def plot_EB_window_all_segs(ds_list, t0, t1, save_dir):
    """全 seg を重ねて [t0, t1] の波形を描いて保存"""

    fig = plt.figure(figsize=(10, 10))
    gs = fig.add_gridspec(6, 1)
    ax_0 = fig.add_subplot(gs[0, 0])
    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
    ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
    ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    ax_1.tick_params(axis='x', which='both', labelbottom=False)
    ax_2.tick_params(axis='x', which='both', labelbottom=False)
    ax_3.tick_params(axis='x', which='both', labelbottom=False)
    ax_4.tick_params(axis='x', which='both', labelbottom=False)

    # ---- 各 seg を同じ軸に重ね描き ----
    for ds_seg in ds_list:
        ds_win = ds_seg.sel(time=slice(t0, t1))
        if ds_win.time.size == 0:
            continue

        ax_0.plot(ds_win.time, ds_win['E8_dsl_x'], lw=1, c='k')
        ax_1.plot(ds_win.time, ds_win['E8_dsl_y'], lw=1, c='k')
        ax_2.plot(ds_win.time, ds_win['E8_dsl_z'], lw=1, c='k')
        ax_3.plot(ds_win.time, ds_win['B8_dsl_x'], lw=1, c='k')
        ax_4.plot(ds_win.time, ds_win['B8_dsl_y'], lw=1, c='k')
        ax_5.plot(ds_win.time, ds_win['B8_dsl_z'], lw=1, c='k')

    ax_0.set_ylabel(r'$E_{x}$ (DSL)' + '\n' + '[mV/m]')
    ax_1.set_ylabel(r'$E_{y}$ (DSL)' + '\n' + '[mV/m]')
    ax_2.set_ylabel(r'$E_{z}$ (DSL)' + '\n' + '[mV/m]')
    ax_3.set_ylabel(r'$B_{x}$ (DSL)' + '\n' + '[nT]')
    ax_4.set_ylabel(r'$B_{y}$ (DSL)' + '\n' + '[nT]')
    ax_5.set_ylabel(r'$B_{z}$ (DSL)' + '\n' + '[nT]')

    for ax in (ax_0, ax_1, ax_2, ax_3, ax_4, ax_5):
        ax.minorticks_on()
        ax.grid(which='both', alpha=0.5)

    ax_5.set_xlim(t0, t1)
    fig.tight_layout()

    if os.path.isdir(save_dir):
        save_dir_   = f'{save_dir}/waveform/{step_min}min_dsl_8Hz'
        os.makedirs(save_dir_, exist_ok=True)
        t0_str = np.datetime_as_string(t0, unit='m').replace('-', '').replace('T', '_').replace(':', '')
        t1_str = np.datetime_as_string(t1, unit='m').split('T')[1].replace(':', '')
        fname = f'EB_fields_dsl_{t0_str}_{t1_str}.png'
        fpath = os.path.join(save_dir_, fname)
        fig.savefig(fpath)
        plt.close(fig)
    else:
        plt.show()
        plt.close()


from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp

def worker(args):
    """並列実行するためのラッパー"""
    ds_list, t0, t1, save_dir = args
    plot_EB_window_all_segs(ds_list, t0, t1, save_dir)
    return str(t0)  # ログ用

# 並列実行用のタスクをまとめる
tasks = [(ds_EB8_dsl_segs, t0, t1, path_base_save_plot) for (t0, t1) in t_list]

# CPUコア数
n_cores = max(1, mp.cpu_count())

with ProcessPoolExecutor(max_workers=n_cores) as exe:
    for out in exe.map(worker, tasks):
        print("done:", out)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams['font.size'] = 15

# ---- 5分刻みの時間窓 ----
t_start = np.datetime64('2022-09-01T20:00:00')
t_end   = np.datetime64('2022-09-02T00:00:00')
step_min    = 1
step    = np.timedelta64(step_min, 'm')

t_list = []
t0 = t_start
while t0 < t_end:
    t1 = t0 + step
    t_list.append((t0, t1))
    t0 = t1

def plot_EB_window_all_segs(ds_list, t0, t1, save_dir):
    """全 seg を重ねて [t0, t1] の波形を描いて保存"""

    fig = plt.figure(figsize=(10, 10))
    gs = fig.add_gridspec(6, 1)
    ax_0 = fig.add_subplot(gs[0, 0])
    ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
    ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
    ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
    ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
    ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    ax_1.tick_params(axis='x', which='both', labelbottom=False)
    ax_2.tick_params(axis='x', which='both', labelbottom=False)
    ax_3.tick_params(axis='x', which='both', labelbottom=False)
    ax_4.tick_params(axis='x', which='both', labelbottom=False)

    # ---- 各 seg を同じ軸に重ね描き ----
    for ds_seg in ds_list:
        ds_win = ds_seg.sel(time=slice(t0, t1))
        if ds_win.time.size == 0:
            continue

        ax_0.plot(ds_win.time, ds_win['E128_dsl_x'], lw=1, c='k')
        ax_1.plot(ds_win.time, ds_win['E128_dsl_y'], lw=1, c='k')
        ax_2.plot(ds_win.time, ds_win['E128_dsl_z'], lw=1, c='k')
        ax_3.plot(ds_win.time, ds_win['B128_dsl_x'], lw=1, c='k')
        ax_4.plot(ds_win.time, ds_win['B128_dsl_y'], lw=1, c='k')
        ax_5.plot(ds_win.time, ds_win['B128_dsl_z'], lw=1, c='k')

    ax_0.set_ylabel(r'$E_{x}$ (DSL)' + '\n' + '[mV/m]')
    ax_1.set_ylabel(r'$E_{y}$ (DSL)' + '\n' + '[mV/m]')
    ax_2.set_ylabel(r'$E_{z}$ (DSL)' + '\n' + '[mV/m]')
    ax_3.set_ylabel(r'$B_{x}$ (DSL)' + '\n' + '[nT]')
    ax_4.set_ylabel(r'$B_{y}$ (DSL)' + '\n' + '[nT]')
    ax_5.set_ylabel(r'$B_{z}$ (DSL)' + '\n' + '[nT]')

    for ax in (ax_0, ax_1, ax_2, ax_3, ax_4, ax_5):
        ax.minorticks_on()
        ax.grid(which='both', alpha=0.5)

    ax_5.set_xlim(t0, t1)
    fig.tight_layout()

    if os.path.isdir(save_dir):
        save_dir_   = f'{save_dir}/waveform/{step_min}min_dsl_128Hz'
        os.makedirs(save_dir_, exist_ok=True)
        t0_str = np.datetime_as_string(t0, unit='m').replace('-', '').replace('T', '_').replace(':', '')
        t1_str = np.datetime_as_string(t1, unit='m').split('T')[1].replace(':', '')
        fname = f'EB_fields_dsl_{t0_str}_{t1_str}.png'
        fpath = os.path.join(save_dir_, fname)
        fig.savefig(fpath)
        plt.close(fig)
    else:
        plt.show()
        plt.close()


from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp

def worker(args):
    """並列実行するためのラッパー"""
    ds_list, t0, t1, save_dir = args
    plot_EB_window_all_segs(ds_list, t0, t1, save_dir)
    return str(t0)  # ログ用

# 並列実行用のタスクをまとめる
tasks = [(ds_EB128_dsl_segs, t0, t1, path_base_save_plot) for (t0, t1) in t_list]

# CPUコア数
n_cores = max(1, mp.cpu_count())

with ProcessPoolExecutor(max_workers=n_cores) as exe:
    for out in exe.map(worker, tasks):
        print("done:", out)